# Exercise - Car engine condition classification using a CNN

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## Load and preprocess data
Dataset: https://www.timeseriesclassification.com/description.php?Dataset=FordA

In [ ]:
def load_ucr_txt(path):
    """
    Args:
        path: filepath of the .txt file to load.
    """
    data = np.loadtxt(path)

    y = data[:, 0]
    X = data[:, 1:]

    return X, y

In [ ]:
# Load data
X_train, y_train = load_ucr_txt("")  # TODO: Add path
X_test, y_test = load_ucr_txt("")   # TODO: Add path

In [ ]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
print("Train class distribution:")
print("Class -1:", np.sum(y_train == -1))
print("Class 1:", np.sum(y_train == 1))

print("\nTest class distribution:")
print("Class -1:", np.sum(y_test == -1))
print("Class 1:", np.sum(y_test == 1))

In [ ]:
print("Train value range:")
print("min:", X_train.min(), "max:", X_train.max())

print("\nTest value range:")
print("min:", X_test.min(), "max:", X_test.max())

In [ ]:
# Pick one random sample per class
idx_neg = np.random.choice(np.where(y_train == -1)[0])
idx_pos = np.random.choice(np.where(y_train == 1)[0])

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(X_train[idx_neg])
plt.title("Class -1")

plt.subplot(1, 2, 2)
plt.plot(X_train[idx_pos])
plt.title("Class 1")

plt.suptitle("Random training examples for both classes")
plt.tight_layout()
plt.show()

## Initialize PyTorch Datasets and Dataloaders

In [ ]:
class FordADataset(Dataset):
    def __init__(self, X, y):
        X = torch.tensor(X, dtype=torch.float32)

        mean = X.mean(dim=1, keepdim=True)
        std = X.std(dim=1, keepdim=True)
        X = (X - mean) / (std + 1e-8)

        X = X.unsqueeze(-1)  # Add a feature dimension

        y = ((torch.tensor(y) + 1) // 2).long()

        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_ds = FordADataset(X_train, y_train)
test_ds = FordADataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

## Define the CNN classifier

In [ ]:
class CNNClassifier(nn.Module):
    def __init__(self, input_channels=1, num_classes=2):
        super().__init__()

        self.cnn = nn.Sequential(
            # TODO: add 1D convolution with kernel size 3, padding 1, and 64 kernels (output channels)
            nn.BatchNorm1d(64, momentum=0.5),
            nn.ReLU(),

            # TODO: add 1D convolution with kernel size 3, padding 1, and 64 kernels (output channels)
            nn.BatchNorm1d(64, momentum=0.5),
            nn.ReLU(),

            # TODO: add 1D convolution with kernel size 3, padding 1, and 64 kernels (output channels)
            nn.BatchNorm1d(64, momentum=0.5),
            nn.ReLU(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        # Input: (B, L, C)
        x = x.transpose(1, 2) # out: (B, C, L); bring data in the format expected by nn.Conv1d

        x = self.cnn(x)
        x = self.pool(x)  # (B, 64, 1)
        x = x.squeeze(-1)  # (B, 64)
        logits = self.fc(x)  # (B, num_classes)

        return logits

In [ ]:
# TODO: Initialize the model

## Train the model

In [ ]:
def train(model, train_loader, num_epochs, device='cpu'):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()

    for epoch in range(num_epochs):
        total_loss = 0.0

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            logits = model(X)

            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

    return model

In [ ]:
# TODO: Train model

## Evaluate the model

In [ ]:
# TODO: Evaluate the model